# Diagnóstico de linearidade — teste de Wang & Phillips (2012)

**Pergunta (slide 29):** a relação de equilíbrio é de fato não linear? $H_0: g(x) = \alpha + \beta x$
contra $H_1: g$ não linear. É a versão formal do que a CV faz informalmente quando escolhe h = ∞.

**Papel no método:** diagnóstico. Não seleciona pares — num par sem cointegração o resíduo linear passeia
e o teste rejeitaria "reta" com força, premiando pares quebrados (DECISOES.md § 8.2). Aqui ele é aplicado
a pares que já passaram num teste de cointegração.

### O teste (Wang & Phillips, 2012, *Annals of Statistics* 40(2), Seção 2 eq. (2) e Teorema 3.1)

$$S_n = \sum_{s \ne t} \hat u_s \hat u_t\, K\!\left(\tfrac{x_t - x_s}{h}\right), \qquad
V_n^2 = \sum_{s \ne t} \hat u_s^2 \hat u_t^2\, K^2\!\left(\tfrac{x_t - x_s}{h}\right), \qquad
Z_n = \frac{S_n}{\sqrt 2\, V_n} \to N(0,1)$$

com $\hat u$ os resíduos do OLS sob $H_0$; rejeita para $Z_n$ grande. Intuição: se a relação for curva, os
resíduos da reta têm o mesmo sinal para preços parecidos, e a soma ponderada pela proximidade em $x$ cresce.

### Adaptações (todas documentadas em DECISOES.md § 9)

- **Kernel** gaussiano (o artigo só exige K limitado e integrável, Assumption 3).
- **Escala:** $x$ dividido pelo desvio do incremento diário, como o passeio do desenho de simulação do
  artigo; $h = n^{-1/3}$, o meio da grade usada por eles ({n^(−1/4), n^(−1/3), n^(−1/2,5)}, Seção 4).
- **Timing** contemporâneo ($y_t = g(x_t) + u_t$), o da cointegração do método; o artigo usa $y_{t+1}$ com $x_t$.
- ⚠️ **Erro autocorrelacionado.** O artigo supõe $u_t$ diferença de martingal (Assumption 2(i)) e diz que a
  extensão a erros serialmente correlacionados não está provada. Nossos spreads têm AR(1) ≈ 0,9. A seção 1
  mostra que, nesse caso, a aproximação N(0,1) rejeita a reta na maioria dos pares **lineares**. Por isso o
  p-valor usado vem de um **bootstrap sob $H_0$ com erro AR(1)**: $y^* = \hat\alpha + \hat\beta x + u^*$,
  $u^*_t = \hat\varphi u^*_{t-1} + e^*_t$, $e^*$ reamostrado; $x$ fixo; OLS e $Z_n$ refeitos em cada réplica.
  É a mesma lógica do valor crítico simulado da formação própria.

In [ ]:
import json, os, time, warnings
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd

from especificacao_np import teste_wp, aplica_lote
from formacao_utils import filtrar_ativos_completos, normalizar_precos

warnings.filterwarnings('ignore')

B          = 199      # réplicas do bootstrap por par
H_EXPOENTE = 1 / 3    # h = n^(−1/3) em unidades de incremento diário
NIVEL      = 0.05
SEMENTE    = 20261102
N_SIM      = 150      # pares simulados por cenário na validação
N_PROCESSOS = 3
SAIDA = 'output/diagnostico'
os.makedirs(SAIDA, exist_ok=True)
META_COLS = ['year', 'semester', 'period', 'day_in_semester', 'total_days_sem']

## 1. Validação: o teste rejeita a reta quando ela é verdadeira?

Pares simulados com $y = 1 + 0{,}9(x-1) + c(x-1)^2 + u$, $x$ um passeio com volatilidade diária de 1,5%, $u$
AR(1) com desvio 0,02. $c = 0$: a reta é verdadeira (a taxa de rejeição deveria ser 5%); $c = -1$: curva
(taxa de rejeição = poder).

In [ ]:
rng = np.random.default_rng(SEMENTE)


def passeio(n=252):
    return np.r_[1, np.cumprod(1 + rng.normal(0.0003, 0.015, n - 1))]


def ar1(n, phi, s=0.02):
    e = rng.normal(0, s * np.sqrt(1 - phi ** 2), n + 100)
    u = np.zeros(n + 100)
    for t in range(1, n + 100):
        u[t] = phi * u[t - 1] + e[t]
    return u[100:]


linhas = []
t0 = time.time()
for phi in (0.0, 0.8, 0.9, 0.95):
    for c in (0.0, -1.0):
        for k in range(N_SIM):
            x = passeio()
            y = 1 + 0.9 * (x - 1) + c * (x - 1) ** 2 + ar1(252, phi)
            r = teste_wp(x, y, B=B, h_expoente=H_EXPOENTE, rng=rng)
            linhas.append({'phi': phi, 'relacao': 'curva' if c else 'reta', 'p_boot': r['p_boot'], 'p_normal': r['p_normal']})
val = pd.DataFrame(linhas)
tab_val = val.groupby(['relacao', 'phi']).agg(rejeita_bootstrap=('p_boot', lambda p: (p < NIVEL).mean()),
                                              rejeita_normal=('p_normal', lambda p: (p < NIVEL).mean()))
val.to_csv(f'{SAIDA}/wp_validacao.csv', index=False)
display(tab_val.style.format('{:.1%}'))
print(f'{time.time() - t0:.0f} s')

## 2. Aplicação aos pares da formação própria

Cada par na direção escolhida pela formação (dependente ~ g(independente)), na janela de formação.

In [ ]:
df = pd.read_csv('../../data_bases/base_unificada.csv', parse_dates=['date']).set_index('date')
meses_linha = pd.PeriodIndex(df.index, freq='M')


def tarefas_de(pares, semente_base):
    tarefas, chaves = [], []
    for k, p in enumerate(pares.itertuples()):
        dep, ind = (p.ativo1, p.ativo2) if p.direcao == 'A~B' else (p.ativo2, p.ativo1)
        mask = (meses_linha >= pd.Period(p.formacao_inicio, 'M')) & (meses_linha <= pd.Period(p.formacao_fim, 'M'))
        pf = df.loc[mask, [dep, ind]].dropna()
        if len(pf) < 100:
            continue
        Q = pf / pf.iloc[0]
        tarefas.append((Q[ind].values, Q[dep].values, B, H_EXPOENTE, semente_base + k))
        chaves.append((p.janela, p.ativo1, p.ativo2))
    return tarefas, chaves


def roda(tarefas):
    lotes = [tarefas[i:i + 200] for i in range(0, len(tarefas), 200)]
    with ProcessPoolExecutor(N_PROCESSOS) as ex:
        return [r for lote in ex.map(aplica_lote, lotes) for r in lote]


propria = pd.read_csv('output/pares/pares_formados_np_propria.csv')
t0 = time.time()
tar, ch = tarefas_de(propria, SEMENTE)
res = pd.DataFrame(roda(tar))
res[['janela', 'ativo1', 'ativo2']] = pd.DataFrame(ch)
wp_prop = propria.merge(res, on=['janela', 'ativo1', 'ativo2'])
wp_prop.to_csv(f'{SAIDA}/wang_phillips_propria.csv', index=False)
print(f"Pares: {len(wp_prop):,} | {time.time() - t0:.0f} s")
print(f"Rejeitam a reta (bootstrap, 5%): {(wp_prop['p_boot'] < NIVEL).mean():.1%}  | "
      f"pela N(0,1) do artigo: {(wp_prop['p_normal'] < NIVEL).mean():.1%}")
print(f"AR(1) do resíduo linear: mediana {wp_prop['phi'].median():.2f}")
display(pd.crosstab(wp_prop['h_linear'].map({True: 'CV escolheu a reta', False: 'CV escolheu curva'}),
                    (wp_prop['p_boot'] < NIVEL).map({True: 'W&P rejeita a reta', False: 'W&P não rejeita'}),
                    normalize='index').style.format('{:.1%}'))

## 3. A não linearidade significativa prevê vantagem do NP?

Nos pares da **formação linear** podemos comparar o NP e a linear par a par (mesmo par, mesma janela).
Pergunta: nos pares em que o teste rejeita a reta, o NP ganha da linear?

In [ ]:
lin_np = pd.read_csv('output/pares/pares_formados_np.csv')     # pares da linear, h e direção da CV unilateral
t0 = time.time()
tar, ch = tarefas_de(lin_np, SEMENTE + 10 ** 6)
res = pd.DataFrame(roda(tar))
res[['janela', 'ativo1', 'ativo2']] = pd.DataFrame(ch)
wp_lin = lin_np.merge(res, on=['janela', 'ativo1', 'ativo2'])
wp_lin.to_csv(f'{SAIDA}/wang_phillips_pares_linear.csv', index=False)
print(f"Pares: {len(wp_lin):,} | {time.time() - t0:.0f} s | rejeitam a reta: {(wp_lin['p_boot'] < NIVEL).mean():.1%}")

K = ['janela', 'ativo1', 'ativo2']
np_res = pd.read_csv('output/resultados/negociacao_pares_np_pares_linear.csv')
lin_res = pd.read_csv('../validation/auditoria/nonparametric/lin_rad_2015_pares.csv')
m = (wp_lin[K + ['p_boot']].merge(np_res[K + ['retorno_total']], on=K)
     .merge(lin_res[K + ['retorno_total']], on=K, suffixes=('_np', '_lin')))
m['dif'] = m['retorno_total_np'] - m['retorno_total_lin']
m['grupo'] = np.where(m['p_boot'] < NIVEL, 'W&P rejeita a reta', 'W&P não rejeita')
tab3 = m.groupby('grupo').agg(pares=('dif', 'size'), ret_np=('retorno_total_np', 'mean'),
                              ret_lin=('retorno_total_lin', 'mean'), np_menos_lin=('dif', 'mean'))
display(tab3.style.format({'pares': '{:,.0f}', 'ret_np': '{:+.4f}', 'ret_lin': '{:+.4f}', 'np_menos_lin': '{:+.4f}'}))
print('Correlação de postos entre o Z_n e a vantagem NP − linear:',
      round(wp_lin.merge(m[K + ['dif']], on=K)[['z', 'dif']].rank().corr().iloc[0, 1], 3))

resumo = {'validacao': {f'{r}_{p}': {'boot': float(v.rejeita_bootstrap), 'normal': float(v.rejeita_normal)}
                        for (r, p), v in tab_val.iterrows()},
          'propria': {'pares': int(len(wp_prop)), 'rejeita_boot': float((wp_prop['p_boot'] < NIVEL).mean()),
                      'rejeita_normal': float((wp_prop['p_normal'] < NIVEL).mean()), 'phi_mediano': float(wp_prop['phi'].median())},
          'pares_linear': {'pares': int(len(wp_lin)), 'rejeita_boot': float((wp_lin['p_boot'] < NIVEL).mean()),
                           'np_menos_lin': {g: float(v) for g, v in tab3['np_menos_lin'].items()}},
          'parametros': {'B': B, 'h_expoente': H_EXPOENTE, 'nivel': NIVEL, 'semente': SEMENTE}}
json.dump(resumo, open(f'{SAIDA}/wang_phillips_resumo.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

### Referências
- Wang, Q. & Phillips, P. C. B. (2012). *A specification test for nonlinear nonstationary models.* Annals of Statistics 40(2). DOI 10.1214/12-AOS975 — estatística (Seção 2, eq. 2), normalização e limite N(0,1) (Teorema 3.1), grade de bandwidth (Seção 4), hipótese de diferença de martingal (Assumption 2(i)).
- Gao, J., King, M., Lu, Z. & Tjøstheim, D. (2009). *Nonparametric specification testing for nonlinear time series with nonstationarity.* Econometric Theory 25 — teste próximo, citado por eles.
- Slides *Cointegração Não Paramétrica via Kernel* (ECO 02091, UFRGS), 29: não linearidade estatística ≠ alpha econômico.